In [1]:
import torch
torch.cuda.init()

In [2]:
import random
import numpy as np
from tqdm import tqdm

from dnallm import load_config
from dnallm import load_model_and_tokenizer, DNAInference, Mutagenesis

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import List, Union, Dict, Tuple

In [4]:
# Load configurations
configs = load_config("./inference_config.yaml")
configs["task"].task_type = "mask"
configs["inference"].max_length = 1024

In [5]:
model_name = "lgq12697/PlantCAD2-Small-l24-d0768"
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs["task"], source="modelscope")

13:14:56 - dnallm.models.model - INFO - Model files are stored in /home/liuguanqing/.cache/modelscope/hub/models/lgq12697/PlantCAD2-Small-l24-d0768


In [6]:
tokenizer

PreTrainedTokenizerFast(name_or_path='/home/liuguanqing/.cache/modelscope/hub/models/lgq12697/PlantCAD2-Small-l24-d0768', vocab_size=7, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'pad_token': '[PAD]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
# Create inference engine
inference_engine = DNAInference(
    model=model,
    tokenizer=tokenizer,
    config=configs
)

13:14:57 - dnallm.models.model - INFO - Using device: cuda


In [ ]:
sequence = (
"GTCGTTTTATTGTAGTTTCTCCGCGTTTGGTACATCAACACTTGTAGCTGTGTATCTTTCTAATTCTTCCTTTCTTTCGC"
"TGACAGTGCTTGCTCACACACTAATCGCAATTGTTGCTGCATTGCCGACTCTGATTCGTTCGATTTCAGCATTTTGTTCT"
"TCCGAAGCGATGCCTCGTCGATCCGGAGATTGCATGAGATGTCTGGTGATTTTCGCGGTTGTATCAGCTTTAGTTGTGTG"
"TGGACCTGCTCTTTATTGGAAATTCAACAAGGGTTTCGTTGGATCCACACGTGCCAACTCTCTTTGTCCTCCTTGCGTTT"
"GTGATTGTCCTCCTCCTTTGTCTCTTCTCCAGATTGCTCCTGGTATGCTTTACCTTTATTCTGATTTGGATCTTACATGT"
"TCCTCGTTTTATGATTAGATCCTTTTACAAGTTTTTCTTGTGAAATTTAGTTCCTTTTCGAGTAATGATCATAATTTTGA"
"GGTGCCCTTTTCCTCGAATTTCGTTTTGCTCTAGTTTATTTGGATCTTTGCTCTGAAGCATCTGATTTTTTCAATATTGG"
"ATCTTTCAATCAAAGCTGTCTCTTTTAGGTTTAATTTCAGTAAGAAAGAATCTCGACTTGTATATTGTTTAACAAGTTGA"
"ATTTTTGTTTTGTTTGCAGGGCTCGCAAATCTCTCTATTACAGGTTAGATCTCGATTCTGAGCTTGACTTTGTTTTAGCT"
"GCTATGCTTGTGTTGTTTCCTTTATTGGTTTTCTATCATGTTGCAATATGGTTTGACAGTTATTGAGAAGCTTTTATTTT"
"GTAAGATTGGTTTCCCATTTGGTATTTGATACCGGAAAATGAATTCTGTCTTGTGTTGTGAACAGATTGTGGAAGCGACG"
"ATCCAGAGCTTAAACAAGAGATGGAGAAGCAATTTGTGGACCTTCTGACTGAGGAGCTGAAGCTGCAAGAAGCAGTTGCA"
"GATGAGCATAGCCGTCATATGAATGTTACCCTTGCAGAAGCTAAAAGGGTTGCATCTCAGTATCAAAAGGAGGCTGAGAA"
"GTGCAATGCCGCCACTGAGATTTGTGAATCAGCTAGAGAGAGAGCTGAAGCTTTGTTGATCAAGGAGAGGAAGATTACTT"
"CTTTGTGGGAGAAGAGAGCTCGGCAATCAGGTTGGGAAGGTGAGTAAACTGATTGCACTTCTCAGTCTACCTTAAAGTGT"
"TTCTCTCTGAGAAATCGAATTGTCACAGCTCTAAAAGGAAATTGTTTAGTTGTCAAGGCAAAGTCCTATCTTCCACGTTT"
"CTCTATATACAGTATGTAGTTTTTTGTTTTGCCCTTGATCGGAATGTAGCTAAAGAAAGGCCTTGTATCAACTTTTATTC"
"ATATATAATTGTCCATTTCTTTACTATGAA"
)

In [107]:
mutagenesis = Mutagenesis(
    model=model, tokenizer=tokenizer, config=configs
)

In [108]:
mutagenesis.mutate_sequence(sequence, replace_mut=False)

Encoding inputs:   0%|          | 0/1 [00:00<?, ? examples/s]

In [109]:
scores = mutagenesis.mlm_evaluate(return_sum=False)

Inferring: 100%|██████████| 1/1 [02:13<00:00, 133.12s/it]


In [131]:
import altair as alt
import pandas as pd
import numpy as np

def plot_token_scatter(
    scores: list[tuple[str, float]],
    threshold_std: float = 2.0,
    show_labels: bool = False,
    extra_data: list[tuple[str, int, int]] | None = None,
    width: int = 800,
    height: int = 300,
    save_path: str | None = None,
) -> alt.Chart:
    """
    使用 Z-score 绘制异常值散点图。

    此函数接受一个 (name, value) 对的列表，计算 Z-score，
    并使用 Altair 绘制散点图，用不同颜色和大小高亮显示异常值。
    这模仿了您提供的 matplotlib 代码的功能。

    Args:
        scores: 格式为 [(name, value), ...] 的数据列表。
        threshold_std: Z-score 阈值，用于定义异常值。
        width: 图表宽度。
        height: 图表高度。
        save_path: 可选的保存图表的 .json 或 .html 路径。

    Returns:
        alt.Chart: 生成的 Altair 图表对象。
    """

    try:
        df = pd.DataFrame([(x[0], -x[1]) for x in scores], columns=["Token", "Value"])
    except Exception as e:
        print(f"Format Error: {e}")
        return alt.Chart()

    # Use index as a base x-axis
    df = df.reset_index()

    mean_val = df["Value"].mean()
    std_val = df["Value"].std()

    # Calculate upper limit for outlier detection
    upper_limit = mean_val + threshold_std * std_val

    # Mark outliers based on Z-score
    df["zscore"] = (df["Value"] - mean_val) / std_val
    # Only consider positive Z-scores for outlier detection
    df["is_outlier"] = df["zscore"] > threshold_std

    # For legend and size encoding
    num_outliers = df["is_outlier"].sum()
    num_normal = len(df) - num_outliers

    df["Status"] = np.where(
        df["is_outlier"],
        f"Outlier (n={num_outliers})",
        f"Normal (n={num_normal})"
    )

    # Base Scatter Layer
    # Use status for color encoding
    base_scatter = alt.Chart(df).mark_point(filled=True).encode(
        x=alt.X("index:Q", title="Token Index"),
        y=alt.Y("Value:Q", title="-LogP Score",
                scale=alt.Scale(domain=(0.0, df["Value"].max() * 1.1))),
        color=alt.Color("Status:N",
            scale=alt.Scale(
                domain=[f"Outlier (n={num_outliers})",
                        f"Normal (n={num_normal})"],
                range=["#fb8072", "#80b1d3"]
            ),
            legend=alt.Legend(title="Importance")
        ),
        # Point size encoding
        size=alt.condition(
            alt.datum.is_outlier,
            alt.value(40),
            alt.value(20)
        ),
        # Alpha encoding
        opacity=alt.condition(
            alt.datum.is_outlier,
            alt.value(1.0),
            alt.value(0.6)
        ),
        tooltip=["index", "Token", "Value", "zscore"]
    )

    # Rule Layers
    line_data = pd.DataFrame([
        {"label": f"Mean ({mean_val:.2f})",
         "value": mean_val, "color": "grey", "dash": [3,3]},
        {"label": f"+{threshold_std}σ ({upper_limit:.2f})",
         "value": upper_limit, "color": "orange", "dash": [3,3]},
    ])

    rule_lines = alt.Chart(line_data).mark_rule(strokeWidth=2).encode(
        y="value:Q",
        color=alt.Color("label:N",
            scale={"domain": line_data["label"].tolist(),
                   "range": line_data["color"].tolist()},
            legend=alt.Legend(title="Threshold")
        ),
        strokeDash=alt.StrokeDash("label:N",
            scale={"domain": line_data["label"].tolist(),
                   "range": line_data["dash"].tolist()},
            legend=None
        ),
    )

    # Text Layer for Outliers
    text_labels = alt.Chart(df).mark_text(
        align="left",
        dx=5,
        color="darkred"
    ).encode(
        x="index:Q",
        y="Value:Q",
        text="Token:N"
    ).transform_filter(
        alt.datum.is_outlier
    )
    if extra_data:
        # Convert to zero started index if necessary
        start_pos = extra_data[0][1]
        for i, (region_type, start, end) in enumerate(extra_data):
            extra_data[i] = (region_type, start - start_pos, end - start_pos)
        extra_df = pd.DataFrame(extra_data, columns=["Type", "Start", "End"])
        # plot gene structure as background bands
        # Intergenic regions: blank
        # UTRs: light grey with height 2
        # CDS/Exons: darker grey with height 4
        # Introns: black with height 1
        band_chart = alt.Chart(extra_df).mark_rect(opacity=0.3).encode(
            x=alt.X("Start:Q", title="Token Index"),
            x2="End:Q",
            color=alt.Color("Type:N",
                scale=alt.Scale(
                    domain=["Intergenic", "5'-UTR", "CDS", "Intron", "3'-UTR"],
                    range=["#ffffff", "#d9d9d9", "#fdb462", "#80b1d3", "#d9d9d9"]
                ),
                legend=alt.Legend(title="Region Type")
            ),
            tooltip=["Type", "Start", "End"]
        )

    # Combine Layers
    chart = (
        base_scatter + rule_lines
    ).resolve_scale(
        color="independent"
    )
    if show_labels:
        chart += text_labels
    if extra_data:
        chart = band_chart + chart
    chart = chart.properties(
        width=width,
        height=height
    ).configure_axis(
        grid=False
    ).interactive()

    if save_path:
        chart.save(save_path)
        print(f"Figure saved to {save_path}")

    return chart

In [132]:
extra_data = [
    ("5'-UTR", 15101295, 15101463),
    ("CDS", 15101463, 15101656),
    ("Intron", 15101656, 15101954),
    ("CDS", 15101954, 15101977),
    ("Intron", 15101977, 15102160),
    ("CDS", 15102160, 15102461),
    ("3'-UTR", 15102461, 15102684),
]
altair_chart = plot_token_scatter(
        scores=scores[0],
        threshold_std=2.0,
        extra_data=extra_data,
        width=800,
        height=200,
    )
altair_chart

alt.LayerChart(...)